In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

volume_path = "/Volumes/dataworkspace/default/data"

# Read Raw Telemetry
raw_telemetry = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{volume_path}/Iot telemetry.csv")
)

total_incoming_records = raw_telemetry.count()
print(f"Total Ingested Records: {total_incoming_records}")

Total Ingested Records: 8


In [0]:
# A. Parse Timestamps
df_staged = raw_telemetry.withColumn("parsed_timestamp", F.to_timestamp("timestamp"))

# B. Apply Rule Flags
df_flagged = (
    df_staged
    # 1. Null Checks on Mandatory Columns
    .withColumn("flag_null_mandatory", 
        F.when(
            F.col("parsed_timestamp").isNull() | 
            F.col("asset_id").isNull() | 
            (F.trim(F.col("asset_id")) == "") |
            F.col("site_id").isNull() | 
            F.col("building_id").isNull(), 
            1
        ).otherwise(0)
    )
    # 2. Schema / Format Violations (Timestamp parsing failure)
    .withColumn("flag_schema_violation", 
        F.when(F.col("parsed_timestamp").isNull() & F.col("timestamp").isNotNull(), 1).otherwise(0)
    )
    # 3. Outlier Detection (Physical Sensor Bounds)
    .withColumn("flag_outlier", 
        F.when(
            (F.col("temperature (°C)") < -20) | (F.col("temperature (°C)") > 120) |
            (F.col("humidity (%)") < 0) | (F.col("humidity (%)") > 100) |
            (F.col("pressure (hPa)") < 700) | (F.col("pressure (hPa)") > 1300) |
            (F.col("vibration (mm/s)") < 0) | (F.col("vibration (mm/s)") > 50) |
            (F.col("power_consumption (kW)") < 0), 
            1
        ).otherwise(0)
    )
    # 4. Late Arriving Data Detection (>7 days lag)
    .withColumn("max_ds_time", F.expr("max(parsed_timestamp) OVER()"))
    .withColumn("flag_late_arriving", 
        F.when(F.col("parsed_timestamp") < (F.col("max_ds_time") - F.expr("INTERVAL 7 DAYS")), 1).otherwise(0)
    )
)

In [0]:
# Calculate Duplicate Count
duplicates_count = df_flagged.count() - df_flagged.dropDuplicates(["timestamp", "asset_id", "sensor_id"]).count()
df_deduped = df_flagged.dropDuplicates(["timestamp", "asset_id", "sensor_id"])

# Tag Validity and Rejection Reason
df_validated = (
    df_deduped
    .withColumn(
        "is_valid",
        F.when(
            (F.col("flag_null_mandatory") == 0) & 
            (F.col("flag_schema_violation") == 0) & 
            (F.col("flag_outlier") == 0), 
            True
        ).otherwise(False)
    )
    .withColumn(
        "rejection_reason",
        F.concat_ws(", ",
            F.when(F.col("flag_null_mandatory") == 1, "NULL_MANDATORY_FIELD"),
            F.when(F.col("flag_schema_violation") == 1, "SCHEMA_TIMESTAMP_INVALID"),
            F.when(F.col("flag_outlier") == 1, "METRIC_OUTLIER_OUT_OF_BOUNDS")
        )
    )
)

In [0]:
import re

def clean_column_names(df):
    for col_name in df.columns:
        # Replace spaces, parentheses, slashes, %, and special chars with underscores
        clean_name = re.sub(r'[ ,;{}()\n\t=%/°]', '_', col_name)
        # Remove duplicate underscores and strip leading/trailing underscores
        clean_name = re.sub(r'_+', '_', clean_name).strip('_')
        df = df.withColumnRenamed(col_name, clean_name)
    return df

# Separate Clean from Quarantine
clean_telemetry = clean_column_names(
    df_validated.filter(F.col("is_valid") == True).drop("is_valid", "rejection_reason")
)

quarantine_telemetry = clean_column_names(
    df_validated.filter(F.col("is_valid") == False)
)

# Write to Unity Catalog tables
clean_telemetry.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.silver_telemetry_clean")
quarantine_telemetry.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("dataworkspace.default.quarantine_telemetry")

print(f"Clean records saved: {clean_telemetry.count()}")
print(f"Quarantined records routed: {quarantine_telemetry.count()}")

Clean records saved: 8
Quarantined records routed: 0


In [0]:
# Generate Quality Summary Table
total_incoming_records = raw_telemetry.count()
total_valid = clean_telemetry.count()
total_quarantined = quarantine_telemetry.count()
duplicates_count = total_incoming_records - df_deduped.count()

quality_report_data = [
    ("Total Records Ingested", total_incoming_records, "100.0%"),
    ("Duplicate Records Dropped", duplicates_count, f"{round((duplicates_count/max(total_incoming_records,1))*100, 2)}%"),
    ("Null / Missing Value Violations", df_validated.filter(F.col("flag_null_mandatory") == 1).count(), "0.0%"),
    ("Schema / Format Violations", df_validated.filter(F.col("flag_schema_violation") == 1).count(), "0.0%"),
    ("Sensor Outlier Violations", df_validated.filter(F.col("flag_outlier") == 1).count(), "0.0%"),
    ("Passed Quality Gate (Clean Data)", total_valid, "100.0%"),
    ("Quarantined Records", total_quarantined, "0.0%")
]

schema_report = StructType([
    StructField("Quality_Check_Metric", StringType(), False),
    StructField("Record_Count", LongType(), False),
    StructField("Pass_Rate_Percentage", StringType(), False)
])

df_quality_report = spark.createDataFrame(quality_report_data, schema=schema_report)
df_quality_report.write.mode("overwrite").saveAsTable("dataworkspace.default.data_quality_summary_report")

display(df_quality_report)

Quality_Check_Metric,Record_Count,Pass_Rate_Percentage
Total Records Ingested,8,100.0%
Duplicate Records Dropped,0,0.0%
Null / Missing Value Violations,0,0.0%
Schema / Format Violations,0,0.0%
Sensor Outlier Violations,0,0.0%
Passed Quality Gate (Clean Data),8,100.0%
Quarantined Records,0,0.0%
